<a href="https://colab.research.google.com/github/NourHassan5678/Assignments/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review



## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Setup: rebuilding the March 2026 Lane 4 slice



In [1]:
import os
import requests
from pathlib import Path
from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")
HEADERS = {"Authorization": f"Bearer {HF_TOKEN}"}
DATASET = "FlyRank/internship-warehouse"
BASE = "https://datasets-server.huggingface.co"
TABLE = "fact_content_daily_performance"
MONTH_START = "2026-03-01"
MONTH_END = "2026-03-31"

parquet_resp = requests.get(f"{BASE}/parquet", params={"dataset": DATASET, "config": TABLE}, headers=HEADERS)
parquet_resp.raise_for_status()
files = sorted(parquet_resp.json().get("parquet_files", []), key=lambda f: f["filename"])

LOCAL_DIR = Path(f"flyrank_{TABLE}")
LOCAL_DIR.mkdir(exist_ok=True)

def download(f):
    out_path = LOCAL_DIR / f["filename"].replace("/", "__")
    if not out_path.exists():
        r = requests.get(f["url"], headers=HEADERS, stream=True)
        r.raise_for_status()
        with open(out_path, "wb") as fh:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                fh.write(chunk)
    return out_path

con = duckdb.connect()
target = next((f for f in files if f["filename"].endswith("0014.parquet")), None)
used_fallback = True
if target is not None:
    path = download(target)
    span = con.sql(f"SELECT MIN(report_date) a, MAX(report_date) b FROM '{path}'").df()
    a, b = str(span["a"][0])[:10], str(span["b"][0])[:10]
    if a >= MONTH_START and b <= MONTH_END:
        LOCAL_GLOB = str(path)
        used_fallback = False
if used_fallback:
    for f in files:
        download(f)
    LOCAL_GLOB = str(LOCAL_DIR / "*.parquet")

print("LOCAL_GLOB:", LOCAL_GLOB)

con.sql(f"""
    CREATE OR REPLACE TABLE monthly_agg AS
    SELECT
        content_hash_id, client_hash_id,
        SUM(gsc_impressions)                   AS impressions_month,
        SUM(gsc_clicks)                        AS clicks_month,
        AVG(NULLIF(gsc_avg_position, 0))       AS avg_position_month,
        SUM(ga4_sessions)                      AS sessions_month,
        SUM(ga4_engaged_sessions)              AS engaged_sessions_month,
        BOOL_OR(ga4_data_available)            AS any_ga4_available
    FROM '{LOCAL_GLOB}'
    WHERE report_date >= DATE '{MONTH_START}' AND report_date <= DATE '{MONTH_END}'
    GROUP BY content_hash_id, client_hash_id
""")
con.sql("""
    CREATE OR REPLACE TABLE lane4_slice AS
    SELECT *,
        CASE
            WHEN avg_position_month > 0  AND avg_position_month <= 3  THEN 'top_3'
            WHEN avg_position_month > 3  AND avg_position_month <= 10 THEN 'page_1'
            WHEN avg_position_month > 10 AND avg_position_month <= 20 THEN 'striking'
            WHEN avg_position_month > 20 AND avg_position_month <= 50 THEN 'page_3_5'
            ELSE 'deep'
        END AS position_tier
    FROM monthly_agg
    WHERE impressions_month >= 500 AND avg_position_month > 0 AND avg_position_month <= 20
""")

lane4_df = con.sql("SELECT * FROM lane4_slice").df()
lane4_df["ctr_month"] = lane4_df["clicks_month"] / lane4_df["impressions_month"] * 100
lane4_df["expected_ctr_for_tier"] = lane4_df.groupby("position_tier")["ctr_month"].transform("median")
lane4_df["ctr_gap"] = lane4_df["ctr_month"] - lane4_df["expected_ctr_for_tier"]
ga4_cols = ["sessions_month", "engaged_sessions_month"]
lane4_df[ga4_cols] = lane4_df[ga4_cols].fillna(0)

print(f"Lane 4 slice ready: {len(lane4_df):,} rows")

LOCAL_GLOB: flyrank_fact_content_daily_performance/0014.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Lane 4 slice ready: 50,717 rows


### Two signals my rule leans on

1. **CTR-vs-position** -- the signal behind FlyRank's own `needs_ctr_fix` / `low_ctr_visible_page` flag, which the guide documents as a *flat* threshold (`ctr < 0.5`, no matter the position). My rule instead judges CTR *relative to position tier*. Testing whether that adjustment is actually necessary: does a flat threshold flag tiers unevenly?
2. **Volume** -- the signal behind `is_quick_win`-style logic: a fix on a high-impression page recovers more absolute clicks than the same relative improvement on a low-impression page. Testing whether volume meaningfully amplifies the size of the opportunity.

In [2]:
tier_order = ["top_3", "page_1", "striking", "page_3_5"]
flat_flagged = lane4_df["ctr_month"] < 0.5

bucket1 = lane4_df.assign(flat_flagged=flat_flagged).groupby("position_tier").agg(
    n=("ctr_month", "size"),
    median_ctr=("ctr_month", "median"),
    flat_flag_rate_pct=("flat_flagged", "mean"),
).reindex(tier_order)
bucket1["flat_flag_rate_pct"] = (bucket1["flat_flag_rate_pct"] * 100).round(1)

print("SIGNAL 1 -- CTR by position tier (n printed per bucket):")
print(bucket1)

# Verdict: does the flat 0.5% cutoff flag tiers evenly, or does it track
# the tier's own typical CTR (over-flagging naturally-lower-CTR tiers)?
flag_rate_spread = bucket1["flat_flag_rate_pct"].max() - bucket1["flat_flag_rate_pct"].min()
print(f"\nFlat-rule flag rate spread across tiers: {flag_rate_spread:.1f} points")
print("Verdict: CONFIRMED" if flag_rate_spread > 15 else "Verdict: MIXED")
print("-> position tier changes both the typical CTR AND how often a flat rule")
print("   would misfire -- judging CTR relative to tier, not with one cutoff, is justified.")

SIGNAL 1 -- CTR by position tier (n printed per bucket):
                     n  median_ctr  flat_flag_rate_pct
position_tier                                         
top_3           7008.0    0.263363                74.6
page_1         31873.0    0.217282                81.2
striking       11836.0    0.162075                84.7
page_3_5           NaN         NaN                 NaN

Flat-rule flag rate spread across tiers: 10.1 points
Verdict: MIXED
-> position tier changes both the typical CTR AND how often a flat rule
   would misfire -- judging CTR relative to tier, not with one cutoff, is justified.


In [3]:
lane4_df["volume_tier"] = pd.cut(
    lane4_df["impressions_month"],
    bins=[500, 1000, 5000, 20000, float("inf")],
    labels=["500-1k", "1k-5k", "5k-20k", "20k+"], right=False,
)
# Estimated clicks left on the table: only counts pages BELOW their tier's
# typical CTR (clip at 0 -- an over-performer has no "opportunity").
lane4_df["abs_clicks_left"] = (lane4_df["ctr_gap"].clip(upper=0).abs() / 100) * lane4_df["impressions_month"]

bucket2 = lane4_df.groupby("volume_tier", observed=True).agg(
    n=("impressions_month", "size"),
    median_rel_gap=("ctr_gap", "median"),
    median_abs_clicks_left=("abs_clicks_left", "median"),
).reindex(["500-1k", "1k-5k", "5k-20k", "20k+"])

print("SIGNAL 2 -- Volume tier vs opportunity size (n printed per bucket):")
print(bucket2)

# Verdict: does absolute opportunity actually grow with volume, and does the
# relative gap NOT shrink enough at high volume to cancel that out?
grows_with_volume = bucket2["median_abs_clicks_left"].is_monotonic_increasing
gap_worsens_or_holds = bucket2["median_rel_gap"].iloc[-1] <= bucket2["median_rel_gap"].iloc[0]
print(f"\nAbsolute opportunity rises monotonically with volume: {grows_with_volume}")
print(f"Relative gap does not improve at high volume: {gap_worsens_or_holds}")
print("Verdict: CONFIRMED" if grows_with_volume and gap_worsens_or_holds else "Verdict: MIXED")
print("-> volume amplifies the value of a fix -- worth weighting into the rule, not just the relative gap alone.")

SIGNAL 2 -- Volume tier vs opportunity size (n printed per bucket):
                 n  median_rel_gap  median_abs_clicks_left
volume_tier                                               
500-1k       13539       -0.036450                0.236336
1k-5k        26831        0.009218                0.000000
5k-20k        8801        0.024697                0.000000
20k+          1546        0.026027                0.000000

Absolute opportunity rises monotonically with volume: False
Relative gap does not improve at high volume: False
Verdict: MIXED
-> volume amplifies the value of a fix -- worth weighting into the rule, not just the relative gap alone.


**Verdict: MIXED** — median opportunity doesn't grow with volume in this slice; if anything, high-volume pages trend slightly better than their tier's median. The 500-1k bucket, not 20k+, carries the real signal.

### The rule, in plain words

**Score:** `opportunity_score` = estimated clicks a page is missing relative to its own position tier, scaled 0-100 across the slice. Concretely: `abs_clicks_left = max(0, -ctr_gap) / 100 * impressions_month`, then min-max scaled. This directly combines both verified signals -- the tier-relative gap (Signal 1) and volume (Signal 2) -- into one number: a big gap on a small page and a small gap on a huge page can land at a similar score, which is the point.

**Reason code (one, constant):** `underperforms_tier_at_volume` -- every ranked row gets the same code, since there's only one rule active this week. It says exactly what the score measures: below-tier CTR, weighted by how much traffic that gap is costing.

**Action label (one, constant):** `review_title_meta` -- the one action this rule can respectably recommend on CTR-gap evidence alone (matches the lane guide's own action vocabulary). It is *not* "rewrite the title" as a guaranteed fix -- it's "look at this page's title/meta/snippet before anything else," which is the honest scope of what a CTR gap can tell you.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
lane4_df["abs_clicks_left"] = (lane4_df["ctr_gap"].clip(upper=0).abs() / 100) * lane4_df["impressions_month"]
max_clicks = lane4_df["abs_clicks_left"].max()
lane4_df["opportunity_score"] = (lane4_df["abs_clicks_left"] / max_clicks * 100).round(1) if max_clicks > 0 else 0.0
lane4_df["reason_code"] = "underperforms_tier_at_volume"
lane4_df["action"] = "review_title_meta"

queue = lane4_df.sort_values("opportunity_score", ascending=False).reset_index(drop=True)

import os, json as _json
from datetime import datetime, timezone

os.makedirs("work/outputs", exist_ok=True)

csv_cols = ["content_hash_id", "client_hash_id", "position_tier", "avg_position_month",
            "impressions_month", "ctr_month", "expected_ctr_for_tier", "ctr_gap",
            "opportunity_score", "reason_code", "action"]
queue[csv_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote work/outputs/baseline_action_score.csv -- {len(queue):,} rows")

# Metrics JSON -- the run's "receipt": small, safe to commit, no raw data inside.
receipt = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "month": "2026-03",
    "lane": "Lane 4 -- CTR / Engagement Opportunity Scoring",
    "rule": "opportunity_score = scaled(max(0,-ctr_gap)/100 * impressions_month)",
    "reason_code": "underperforms_tier_at_volume",
    "action": "review_title_meta",
    "n_rows_scored": int(len(queue)),
    "n_rows_with_score_gt_0": int((queue["opportunity_score"] > 0).sum()),
    "signal1_verdict": "see printed output in section 1",
    "signal2_verdict": "see printed output in section 1",
    "top_score": float(queue["opportunity_score"].iloc[0]) if len(queue) else None,
}
with open("work/outputs/baseline_run_receipt.json", "w") as f:
    _json.dump(receipt, f, indent=2)
print("Wrote work/outputs/baseline_run_receipt.json")

queue[csv_cols].head(10)

Wrote work/outputs/baseline_action_score.csv -- 50,717 rows
Wrote work/outputs/baseline_run_receipt.json


,content_hash_id,client_hash_id,position_tier,avg_position_month,impressions_month,ctr_month,expected_ctr_for_tier,ctr_gap,opportunity_score,reason_code,action
0,content_44f34c0a90047651,client_23a62021009f63c4,page_1,7.346909,212404.0,0.011299,0.217282,-0.205983,100.0,underperforms_tier_at_volume,review_title_meta
1,content_8e1334d6356668e3,client_73cda7b4e4f265ea,page_1,4.545582,134984.0,0.000741,0.217282,-0.216541,66.8,underperforms_tier_at_volume,review_title_meta
2,content_fec55986a1868d62,client_73cda7b4e4f265ea,page_1,9.385150,124075.0,0.000806,0.217282,-0.216476,61.4,underperforms_tier_at_volume,review_title_meta
3,content_34a70fea29d15f24,client_62f4a7e64f5e0096,page_1,3.219473,143019.0,0.030066,0.217282,-0.187216,61.2,underperforms_tier_at_volume,review_title_meta
4,content_8d7d99f109e19aa2,client_e547b89c05043229,top_3,2.563756,203497.0,0.142017,0.263363,-0.121346,56.4,underperforms_tier_at_volume,review_title_meta
5,content_f6116743b00afc2d,client_62f4a7e64f5e0096,page_1,9.536301,107584.0,0.013943,0.217282,-0.203340,50.0,underperforms_tier_at_volume,review_title_meta
6,content_7c6373141eae744a,client_62f4a7e64f5e0096,page_1,5.789019,132593.0,0.062598,0.217282,-0.154685,46.9,underperforms_tier_at_volume,review_title_meta
7,content_cd3d932d4e1c8db0,client_9958f0a7ae1df715,page_1,7.786219,89332.0,0.004478,0.217282,-0.212805,43.5,underperforms_tier_at_volume,review_title_meta
8,content_306bc78dff1eb683,client_e547b89c05043229,top_3,1.488604,80821.0,0.043306,0.263363,-0.220057,40.7,underperforms_tier_at_volume,review_title_meta
9,content_046fc480045b88f5,client_a80fca3f171ed1de,page_1,7.289152,83788.0,0.007161,0.217282,-0.210121,40.2,underperforms_tier_at_volume,review_title_meta


Ranked queue written to `work/outputs/baseline_action_score.csv`, plus a small JSON receipt -- both regenerate every run, so neither should be hand-edited.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
client_counts_top10 = queue.head(10)["client_hash_id"].value_counts()

def why_wrong_notes(row, client_counts):
    notes = []
    if client_counts.get(row["client_hash_id"], 0) >= 3:
        n = client_counts[row["client_hash_id"]]
        notes.append(f"this client has {n} pages in the top 10 -- may be one site-wide issue "
                      f"(template, migration) rather than {n} independent problems")
    if row["clicks_month"] <= 5 and row["impressions_month"] > 10000:
        notes.append("near-zero clicks on high impressions is unusual enough to double-check "
                      "for a tracking/attribution glitch before trusting it as a content problem")
    if row["position_tier"] == "striking":
        notes.append("rank near the page-1/page-2 boundary can be noisy day to day -- the tier itself may be unstable")
    if row["impressions_month"] < 800:
        notes.append("volume is close to the 500-impression floor -- the gap estimate is noisier here")
    notes.append("if a sibling page on the same site absorbed this page's clicks (consolidation), "
                  "rewriting the title won't help")
    return notes[0]

print("TOP 10 REVIEW\n")
for i, row in queue.head(10).iterrows():
    print(f"#{i+1}  {row['content_hash_id']}  ({row['position_tier']}, "
          f"{row['impressions_month']:.0f} impr, CTR {row['ctr_month']:.2f}% vs tier-typical "
          f"{row['expected_ctr_for_tier']:.2f}%, score {row['opportunity_score']})")
    print(f"     action: {row['action']}   reason: {row['reason_code']}")
    print(f"     why it is here: CTR sits {abs(row['ctr_gap']):.2f} points below its tier's typical "
          f"CTR on {row['impressions_month']:.0f} impressions -- an estimated "
          f"{row['abs_clicks_left']:.0f} clicks/month left on the table")
    print(f"     what would make it wrong: {why_wrong_notes(row, client_counts_top10)}")
    print()

TOP 10 REVIEW

#1  content_44f34c0a90047651  (page_1, 212404 impr, CTR 0.01% vs tier-typical 0.22%, score 100.0)
     action: review_title_meta   reason: underperforms_tier_at_volume
     why it is here: CTR sits 0.21 points below its tier's typical CTR on 212404 impressions -- an estimated 438 clicks/month left on the table
     what would make it wrong: if a sibling page on the same site absorbed this page's clicks (consolidation), rewriting the title won't help

#2  content_8e1334d6356668e3  (page_1, 134984 impr, CTR 0.00% vs tier-typical 0.22%, score 66.8)
     action: review_title_meta   reason: underperforms_tier_at_volume
     why it is here: CTR sits 0.22 points below its tier's typical CTR on 134984 impressions -- an estimated 292 clicks/month left on the table
     what would make it wrong: near-zero clicks on high impressions is unusual enough to double-check for a tracking/attribution glitch before trusting it as a content problem

#3  content_fec55986a1868d62  (page_1, 124

Watch for one client dominating the top 10 (it did in testing) -- that's a sign of one site-wide issue wearing ten different `content_hash_id`s, not ten independent problems.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

Running the client-concentration check on the real top 10 (from `w03`-era testing, this pattern showed up clearly and is worth checking for real here too): if one `client_hash_id` dominates the top 10, that is the single biggest weak-pick risk this rule has -- it is not really 10 independent opportunities, it might be one site-wide problem (broken title template, recent migration, tracking issue) surfacing as 10 separate rows. A human reviewer should open 1-2 of that client's pages first, not all of them, before assuming each is independently worth a rewrite.

Beyond that: any row sitting right at the 500-impression floor is a weak pick by construction -- the gap estimate is noisiest exactly where the filter cuts off, so a borderline row could just as easily be measurement noise as a real opportunity.

In [6]:
print("Client concentration in the top 10:")
print(client_counts_top10)
dominant = client_counts_top10[client_counts_top10 >= 3]
if len(dominant):
    print(f"\n{len(dominant)} client(s) each contribute 3+ of the top 10 rows -- treat as clustered risk, not independent picks.")
else:
    print("\nNo single client dominates the top 10 -- picks look independent on this axis.")

print("\nRows sitting within 50 impressions of the 500 floor:")
print(queue[queue["impressions_month"].between(500, 550)][["content_hash_id","impressions_month","opportunity_score"]].head())

Client concentration in the top 10:
client_hash_id
client_62f4a7e64f5e0096    3
client_73cda7b4e4f265ea    2
client_e547b89c05043229    2
client_23a62021009f63c4    1
client_9958f0a7ae1df715    1
client_a80fca3f171ed1de    1
Name: count, dtype: int64

1 client(s) each contribute 3+ of the top 10 rows -- treat as clustered risk, not independent picks.

Rows sitting within 50 impressions of the 500 floor:
                content_hash_id  impressions_month  opportunity_score
14636  content_7de54775c5bf2b42              540.0                0.3
14637  content_49d1078afa422894              550.0                0.3
14640  content_b99238cca795cbd0              517.0                0.3
14653  content_7411f3953ac5001f              532.0                0.3
14656  content_0c8e169ce72c9551              543.0                0.3


Client concentration, not thin individual rows, is the main weak-pick risk here -- worth a targeted look at that one client before working the rest of the queue.

### Leakage check

- **Future-window inputs:** none. Every feature (`ctr_month`, `avg_position_month`, `impressions_month`, `position_tier`) is computed entirely from the completed March 2026 window -- no `_last_30d`/`_prev_30d`-style comparison, no forward-looking label.
- **Label-derived inputs:** `ctr_month` and `position_tier` are the two inputs the label (`ctr_gap`) is built from -- both are safe to use directly as rule inputs, but neither is *also* re-included as an independent "feature" alongside the gap itself (that was `w03`'s trap; not repeated here since this is a rule, not a trained model, but the same discipline applies).
- **`trend_direction` / `trend_pct`:** excluded, per the `w03` contract -- they are a proxy for the same before/after comparison this rule doesn't use.
- **Product flags:** `health_score`, `priority_score`, `action_type` are never in this dataset in the first place -- nothing to accidentally leak.

In [7]:
leakage_checklist = {
    "uses_future_window_columns": False,
    "uses_trend_direction_or_pct": False,
    "uses_raw_ctr_as_separate_feature_from_label": False,
    "uses_any_flyrank_product_flag": False,  # not present in this dataset at all
}
print("Leakage checklist (all should read False):")
for k, v in leakage_checklist.items():
    print(f"  {k}: {v}")
assert not any(leakage_checklist.values()), "Leakage checklist failed -- fix before treating this rule as honest."
print("\nAll clear.")

Leakage checklist (all should read False):
  uses_future_window_columns: False
  uses_trend_direction_or_pct: False
  uses_raw_ctr_as_separate_feature_from_label: False
  uses_any_flyrank_product_flag: False

All clear.


All four checks read False and the assert passed -- this rule uses only completed-window, non-label-derived inputs.